# Itemset on Google Colab

Run all cells on a CPU runtime. The notebook resolves the public repository to an exact commit, verifies Julia 1.13.2, Java 25 and the pinned SPMF JAR, then runs the Julia demo. Setup and application data downloads can take several minutes.

The demo checks the package and compares both variants with an independent SPMF run on Chess. The Chess check measures peak process RSS for one threshold. The full benchmark series have their own commands and figures in the [report guide](https://github.com/NgTHung/Itemset/tree/main/docs/report).

The two applications run in their own notebooks, [market_basket.ipynb](https://colab.research.google.com/github/NgTHung/Itemset/blob/main/notebooks/market_basket.ipynb) and [image_patterns.ipynb](https://colab.research.google.com/github/NgTHung/Itemset/blob/main/notebooks/image_patterns.ipynb). Use the Colab Files sidebar to download this notebook's evidence ZIP.

In [ ]:
import os
from pathlib import Path
import re
import runpy
import subprocess
import tempfile
import urllib.request

ITEMSET_REVISION = ""
repository = os.environ.get("ITEMSET_REPOSITORY", "https://github.com/NgTHung/Itemset")
match = re.fullmatch(r"https://github\.com/([A-Za-z0-9-]+)/([A-Za-z0-9_.-]+)/?", repository)
if not match or match[2].removesuffix(".git") in ("", ".", ".."):
    raise ValueError("ITEMSET_REPOSITORY must be a GitHub HTTPS clone URL (https://github.com/owner/repo)")
repository_slug = f"{match[1]}/{match[2].removesuffix('.git')}"
revision = ITEMSET_REVISION or os.environ.get("ITEMSET_REVISION")
if not revision:
    revision = subprocess.check_output(
        ["git", "ls-remote", "--exit-code", repository, "HEAD"],
        text=True, timeout=60,
    ).split()[0]
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("ITEMSET_REVISION must be a full lowercase 40-character commit SHA")

workspace = Path(tempfile.mkdtemp(prefix="itemset-colab-", dir="/content"))
runner_path = workspace / "run_colab_notebook.py"
runner_url = f"https://raw.githubusercontent.com/{repository_slug}/{revision}/scripts/run_colab_notebook.py"
with urllib.request.urlopen(runner_url, timeout=60) as response:
    runner_path.write_bytes(response.read())
evidence = workspace / "evidence"
print(f"Revision: {revision}", flush=True)
runner = runpy.run_path(str(runner_path))
if runner["main"]([
    "--repository", repository,
    "--revision", revision,
    "--root", str(workspace / "repo"),
    "--evidence", str(evidence),
]):
    raise RuntimeError("Demo failed. See the evidence archive in the cell output.")